# Simple RAG example using Facebook AI Similarity Search (FAISS)

In this example, we'll demonstrate how to use [**FAISS**](https://faiss.ai/) for similarity-based document retrieval. We will simulate a small **mock dataset** of fictional documents and use an **embedding model in Aitta** to encode them into vectors. We will then build a **FAISS index** to enable fast and efficient similarity search. Finally, we will simulate a **RAG** system where we retrieve the most relevant documents and use them to generate an answer with a chat model in Aitta.


## Steps Overview:
1. **Create a mock dataset**: We create a small set of fictional documents.
2. **Generate embeddings**: We use the embedding model `intfloat/multilingual-e5-large` in Aitta to convert these documents into vector embeddings.
3. **Build FAISS index**: We build a **FAISS index** that will store these vector embeddings for fast similarity search.
4. **Search and retrieve**: We perform a similarity search based on a query and retrieve the most relevant document(s).
5. **Answer generation**: Using the retrieved document(s), we simulate a **RAG pipeline** to generate a response with the chat model `LumiOpen/Llama-Poro-2-70B-Instruct` in Aitta.

This example demonstrates how **FAISS** can be used for efficient document retrieval, and how **RAG** can help generate contextually relevant answers from these documents.

## Setup

We use two models through Aitta's OpenAI-compatible API, both running on the LUMI supercomputer:

* an **embedding model**, [`intfloat/multilingual-e5-large`](https://huggingface.co/intfloat/multilingual-e5-large), which turns text into vectors, and
* a **chat model**, [`LumiOpen/Llama-Poro-2-70B-Instruct`](https://huggingface.co/LumiOpen/Llama-Poro-2-70B-Instruct), which generates the answer.

Both are used with the same `openai` client: embeddings with `client.embeddings.create()` and answers with `client.chat.completions.create()`.

In [2]:
# Set genereated access token here  
access_token = ""

In [6]:
import openai

# Configure the standard OpenAI client which is used for both embeddings and chat completions.
client = openai.OpenAI(
    api_key=access_token,
    base_url="https://aitta-api.csc.fi/openai/v1"
)

embedding_model = "intfloat/multilingual-e5-large"

chat_model = "LumiOpen/Llama-Poro-2-70B-Instruct"

## A helper function for creating embeddings

The model `intfloat/multilingual-e5-large` has been trained so that every input text starts with a **prefix** that tells what kind of text it is:

* `"passage: "` for the texts we search from (the documents or chunks), and
* `"query: "` for the search queries (the user's questions).

Without the prefixes, the search results get worse. Other embedding models may use different prefixes or none, so always check the model card.

The function below adds the prefix and sends all texts to Aitta in **one request**, which is faster than sending them one by one.

FAISS needs the vectors as a NumPy array of 32-bit floats, so the function also converts the result into that format.

In [7]:
import numpy as np

def embed(texts, prefix):
    """Create embeddings for a list of texts with the embedding model in Aitta."""
    response = client.embeddings.create(
        model=embedding_model,
        input=[prefix + text for text in texts]
    )
    # One embedding for each input text, as a NumPy array of 32-bit floats for FAISS
    return np.array([item.embedding for item in response.data], dtype="float32")

## Create a mock dataset and generate embeddings

In [8]:
# Create a mock dataset as a list of "documents"
documents = ["Cacapapadadas are grey, 10cm long worms.",
"The moon is actually made of a soft cheese."]


# We don't chunk our documents since they are short in this example.

In [ ]:
# Generate vector embeddings for the documents (note the "passage: " prefix)
vectors = embed(documents, prefix="passage: ")

vectors.shape  # (number of documents, embedding dimension)

## Build a FAISS index

In [ ]:
import faiss

# Determine the dimensionality of the vector embeddings
vector_dimension = vectors.shape[1]

# Initialize FAISS index using the Inner Product (IP) method for cosine similarity search
index = faiss.IndexFlatIP(vector_dimension)
# Alternatively, you could use IndexFlatL2 for Euclidean distance-based similarity

# Normalize the vectors so that the inner product equals cosine similarity
faiss.normalize_L2(vectors)

# Add the vectors to the FAISS index
index.add(vectors)

print("Number of vectors in the index:", index.ntotal)

## Search and retrieve

Now we embed a query with the prefix `"query: "` and search for the most similar documents in the index.

> **Note:** With `intfloat/multilingual-e5-large`, the similarity scores are typically all between about 0.7 and 1.0, even for texts that are not related. This is expected behaviour of this model. What matters is the **order** of the scores: the document with the highest score is the most relevant one.

In [ ]:
# Define the query text for searching in the FAISS index
search_text = "What is the moon made of?"

# Convert the query text into an embedding and normalize it
search_vector = embed([search_text], prefix="query: ")
faiss.normalize_L2(search_vector)

# Perform a search in the FAISS index
k = index.ntotal  # We set k to the total number of documents to see how similar all are to the query
similarities, indices = index.search(search_vector, k=k)

# Print the similarity scores and corresponding indices of the retrieved documents
print(similarities)
print(indices)

In [ ]:
# Print each of the retrieved documents along with its similarity score
for i, idx in enumerate(indices[0]):
    print(f"Rank {i+1}:")
    print("Text:", documents[idx])  # Retrieve the document text by its index
    print("Similarity:", similarities[0][i])  # Inner product of normalized vectors = cosine similarity (higher means more similar)
    print("-" * 50)

## Answer generation with RAG

In [ ]:
input_query = "What are Cacapapadadas?"

# Embed the query and normalize it
query_embedding = embed([input_query], prefix="query: ")
faiss.normalize_L2(query_embedding)

# Perform similarity search on the FAISS index
k = 1  # Number of nearest neighbors to retrieve
similarities, indices = index.search(query_embedding, k)

# Retrieve the document(s) corresponding to the top index
retrieved_documents = [documents[i] for i in indices[0]]
print(retrieved_documents)

print("Most similar document index:", indices)
print("Similarity:", similarities)

# Prepare the prompt
prompt = f"Given the following document, answer the question:\n\nDocument: {retrieved_documents}\n\nQuestion: {input_query}\nAnswer:"

In [ ]:
prompt

In [ ]:
# Generate the answer using the retrieved document
response = client.chat.completions.create(
    messages=[
        {
            "role": "user",
            "content": prompt
        }
    ],
    model=chat_model
)

# Display the answer
answer = response.choices[0].message.content
print("Answer:", answer)

## LLM usage without RAG

Now, let's test how the model responds to the query without relying on an external data source.

In [ ]:
input_query = "What are Cacapapadadas?"

response = client.chat.completions.create(
    messages=[
        {
            "role": "user",
            "content": input_query
        }
    ],
    model=chat_model
)

# Display the answer
answer = response.choices[0].message.content
print("Answer:", answer)

## Did the model hallucinate? 

You may notice that the model generates a response based on patterns in the training data, which could be inaccurate. To reduce the chances of hallucination without utilizing RAG, we can try to provide a more specific prompt.

In [ ]:
input_query = "What are Cacapapadadas?"

prompt = f"Answer the query only if you know the answer for sure. Do not make up any new information. Query: {input_query}"

response = client.chat.completions.create(
    messages=[
        {
            "role": "user",
            "content": prompt
        }
    ],
    model=chat_model
)

# Display the answer
answer = response.choices[0].message.content
print("Answer:", answer)